In [30]:

import matplotlib.pyplot as plt
import numpy as np
from io import BytesIO
from PIL import Image
def segment_to_waveform_image(segment, fs, size=(256, 256)):
    import matplotlib.pyplot as plt
    from io import BytesIO
    from PIL import Image
    import numpy as np

    # Normalize amplitude just in case
    if np.max(np.abs(segment)) > 0:
        segment = segment / np.max(np.abs(segment))

    # Bigger figure + higher DPI so the waveform has detail
    fig, ax = plt.subplots(figsize=(4, 4), dpi=200)
    ax.plot(segment, color='black', linewidth=0.6)
    ax.set_xlim(0, len(segment))
    ax.set_ylim(-1, 1)
    ax.axis('off')

    buf = BytesIO()
    # Don't crop! Cropping destroys scale and causes inconsistent images
    plt.savefig(buf, format='png', bbox_inches=None, pad_inches=0)
    plt.close(fig)

    buf.seek(0)
    img = Image.open(buf).convert('RGB')

    # Downsize *after* high-DPI render → crisp small image
    img = img.resize(size, Image.LANCZOS)
    return np.array(img)



In [31]:
import librosa

def segment_to_mel_spectrogram(segment, fs, size=(128, 128)):
    mel_spec = librosa.feature.melspectrogram(y=segment, sr=fs, n_mels=size[0])
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    img = Image.fromarray(mel_spec_db)
    img = img.resize(size)
    img = np.stack([img, img, img], axis=-1)  # make RGB-like shape
    return img


In [ ]:
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
import librosa
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, BatchNormalization, Flatten, Dense

data = "./features-relative.csv"

df = pd.read_csv(data)
print(repr(df.columns.tolist()))
df["segment"] = None
for index, row in df.iterrows():
    start_sample = int(row[" Start Sample"])
    end_sample = int(row[" End Sample"])
    signal, fs= librosa.load(row["file"])
    segment = signal[start_sample:end_sample]
    df.at[index, "segment"] = segment  

max_len = max(len(segment) for segment in df["segment"]) 
max_len = 100
def pad_segment(segment, max_len):
    if len(segment) > max_len:
        return segment[:max_len]           
    else:
        return np.pad(segment, (0, max_len - len(segment)))  

segments = np.array(df["segment"])
waveforms = np.array([segment_to_waveform_image(segment, fs) for segment in segments])
if waveforms.dtype == np.uint8:
    waveforms = waveforms.astype('float32') / 255.0

# Convert to grayscale using ITU-R 601-2 luma weighting (better than naive mean)
wave_gray = (0.2989 * waveforms[..., 0]
             + 0.5870 * waveforms[..., 1]
             + 0.1140 * waveforms[..., 2])

# wave_gray now is (N, H, W); add channel dimension:
X = wave_gray[..., np.newaxis].astype('float32')   # shape (N, H, W, 1)
print("New X shape:", X.shape, "dtype:", X.dtype)


y = df[' Label'] 

le = LabelEncoder()
y = le.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)


['Start', ' End', ' Label', ' Start Sample', ' End Sample', ' len_1', 'mean_1', 'std_1', 'rms_1', 'mav_1', 'iEMG_1', 'wl_1', 'zcr_1', 'spec_entropy_1', 'median_freq_1', 'skew_1', 'kurtosis_1', 'len_2', 'mean_2', 'std_2', 'rms_2', 'mav_2', 'iEMG_2', 'wl_2', 'zcr_2', 'spec_entropy_2', 'median_freq_2', 'skew_2', 'kurtosis_2', 'len_3', 'mean_3', 'std_3', 'rms_3', 'mav_3', 'iEMG_3', 'wl_3', 'zcr_3', 'spec_entropy_3', 'median_freq_3', 'skew_3', 'kurtosis_3', 'len_4', 'mean_4', 'std_4', 'rms_4', 'mav_4', 'iEMG_4', 'wl_4', 'zcr_4', 'spec_entropy_4', 'median_freq_4', 'skew_4', 'kurtosis_4', 'len_5', 'mean_5', 'std_5', 'rms_5', 'mav_5', 'iEMG_5', 'wl_5', 'zcr_5', 'spec_entropy_5', 'median_freq_5', 'skew_5', 'kurtosis_5', 'len_6', 'mean_6', 'std_6', 'rms_6', 'mav_6', 'iEMG_6', 'wl_6', 'zcr_6', 'spec_entropy_6', 'median_freq_6', 'skew_6', 'kurtosis_6', 'len_7', 'mean_7', 'std_7', 'rms_7', 'mav_7', 'iEMG_7', 'wl_7', 'zcr_7', 'spec_entropy_7', 'median_freq_7', 'skew_7', 'kurtosis_7', 'len_8', 'mean_

KeyError: 'Start Sample'

In [ ]:
# Save waveform images (with labels) to a folder for inspection
import os
from PIL import Image
import pandas as pd

out_dir = "./waveform_samples"
os.makedirs(out_dir, exist_ok=True)

# Number to save (set to None to save all)
SAVE_N = 200
n_samples = len(waveforms)
if SAVE_N is None:
    SAVE_N = n_samples
else:
    SAVE_N = min(SAVE_N, n_samples)

filenames = []
labels_out = []
indices = []

for i in range(SAVE_N):
    img = waveforms[i]
    label = y[i]

    # Convert back to human label if encoder exists
    try:
        human_label = le.inverse_transform([label])[0]
    except Exception:
        human_label = str(label)

    # Extract metadata from DataFrame at this index
    row = df.iloc[i]
    start_sample = int(row["Start"])
    end_sample = int(row[" End"])
    original_file = row["file"]
    # Extract just the filename without path
    original_filename = os.path.basename(original_file).replace('.wav', '')

    # Create filename with start time, end time, original filename, and label
    fname = f"{original_filename}_{start_sample}-{end_sample}_{human_label}.png"
    fpath = os.path.join(out_dir, fname)

    # img should be uint8 in 0-255 already if you normalized earlier; ensure correct dtype
    if img.dtype != np.uint8:
        img_to_save = (img * 255).astype('uint8')
    else:
        img_to_save = img

    Image.fromarray(img_to_save).save(fpath)

    filenames.append(fname)
    labels_out.append(human_label)
    indices.append(i)

# Save mapping CSV
mapping_df = pd.DataFrame({
    "filename": filenames, 
    "label": labels_out, 
    "index": indices,
    "original_file": [df.iloc[i]["file"] for i in indices],
    "start_sample": [int(df.iloc[i][" Start Sample"]) for i in indices],
    "end_sample": [int(df.iloc[i][" End Sample"]) for i in indices]
})
mapping_df.to_csv(os.path.join(out_dir, "mapping.csv"), index=False)

print(f"Saved {len(filenames)} waveform images to {out_dir} and mapping.csv")

Saved 200 waveform images to ./waveform_samples and mapping.csv


In [ ]:

from tensorflow.keras.layers import LSTM

from sklearn.utils.class_weight import compute_class_weight

cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(cw))

model = Sequential([
    Conv2D(32, 7, activation='relu', padding='same', input_shape=(256, 256, 1)),  # Updated input shape
    MaxPooling2D(2),
    BatchNormalization(),
    Conv2D(64, 5, activation='relu', padding='same'),
    MaxPooling2D(2),
    BatchNormalization(),
    Conv2D(64, 3, activation='relu', padding='same'),
    MaxPooling2D(2),
    Dropout(0.3),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])


c:\Users\joaqu\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['acc'])

monitor = EarlyStopping(monitor='val_loss',
                        mode='min',  # Changed from 'max' to 'min' since we're monitoring loss
                        restore_best_weights=True,
                        patience=5)

In [ ]:
history = model.fit(X_train, y_train,
                    epochs=100,
                    validation_split=0.1,
                    callbacks=[monitor],
                    class_weight=class_weights)

Epoch 1/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 232s 3s/step - acc: 0.3450 - loss: 2.0685 - val_acc: 0.5177 - val_loss: 1.0985
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 232s 3s/step - acc: 0.3450 - loss: 2.0685 - val_acc: 0.5177 - val_loss: 1.0985
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - acc: 0.5190 - loss: 1.0949 - val_acc: 0.5177 - val_loss: 1.0984
Epoch 3/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 106s 1s/step - acc: 0.5190 - loss: 1.0949 - val_acc: 0.5177 - val_loss: 1.0984
Epoch 3/100
41/80 ━━━━━━━━━━━━━━━━━━━━ 54s 1s/step - acc: 0.5111 - loss: 1.0524

KeyboardInterrupt: 